# 05 — Integrated Model: Life Satisfaction Among Foreign Residents in Seoul

**Purpose.** Combine prespecified thematic feature families from notebooks 02 (language/integration), 03 (economic/living), and 04 (health/discrimination) under the same fixed experimental design. Model selection occurs only via training cross-validation. The held-out test has already been examined in earlier thematic analyses, so this test is **not a pristine, untouched external validation sample**. This limitation must be disclosed.

This is a predictive comparison on observational survey data; it cannot establish causality. Review `PROJECT_CONTRACT.md`, `MODEL_CONFIG.md`, and `findings/METHODOLOGY_CHANGELOG.md` before interpreting results.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'data' / 'processed' / 'seoul_foreigners_modeling_master.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run 01_preprocessing.ipynb to create the modeling master.')
sys.path.insert(0, str(ROOT / 'src'))
from seoul_foreigners_ml.team_experiments import (
    load_shared_data, validate_features, cv_compare,
    evaluate_winner, feature_permutation, RANDOM_STATE, CV_FOLDS
)
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Load and validate the common split

In [2]:
root, df, train_df, test_df = load_shared_data(ROOT)
print('Full dataset:', df.shape)
print('Training set:', train_df.shape)
print('Test set:', test_df.shape)
print('Classification prevalence (train/test):', train_df.high_satisfaction.mean(), test_df.high_satisfaction.mean())

Full dataset: (2500, 165)
Training set: (2000, 165)
Test set: (500, 165)
Classification prevalence (train/test): 0.655 0.656


## 2. Prespecified integrated feature families

The feature families are taken from the initial research scope and codebook, rather than selected by comparing the thematic test scores. We keep `korean_proficiency_10` rather than simultaneously entering all four source language skills; we use log residence duration rather than both original and log versions. We exclude `ID`, targets, and target-proximal happiness/satisfaction items.

In [3]:
NOMINAL = ['SQ1', 'SQ2', 'SQ7', 'SQ5', 'SQ6', 'DQ1', 'DQ2', 'DQ3', 'DQ4_2']
ORDINAL = [
    'Q19_1','Q19_2','Q19_3','Q19_4',
    'Q20_1','Q20_2','Q20_3','Q20_4',
    'DQ4_1','Q15',
    *[f'Q6_{i}' for i in range(1, 13)],
    *[f'Q24_{i}' for i in range(1, 6)]
]
NUMERIC = ['age', 'months_in_korea_log1p', 'korean_proficiency_10', 'SQ6_1_log1p', 'SQ6_2_log1p']
FEATURES = validate_features(df, NOMINAL, ORDINAL, NUMERIC)
feature_quality = pd.DataFrame({
    'feature': FEATURES,
    'missing_pct': (train_df[FEATURES].isna().mean() * 100).values,
    'n_unique': train_df[FEATURES].nunique(dropna=True).values
})
display(feature_quality.sort_values('missing_pct', ascending=False))

,feature,missing_pct,n_unique
20,Q6_2,51.80,5
6,DQ2,32.10,7
7,DQ3,32.10,18
40,SQ6_2_log1p,32.00,64
39,SQ6_1_log1p,31.05,79
26,Q6_8,15.95,5
35,Q24_5,8.60,5
33,Q24_3,3.55,5
34,Q24_4,3.40,5
24,Q6_6,3.20,5


## 3. Training-only model comparison

Regression is selected by lowest mean 5-fold CV RMSE. Classification is selected by highest 5-fold CV macro-F1. Dummy models establish naive baselines. All preprocessing is fitted *within* each training fold.

In [4]:
reg_cv = cv_compare(train_df, NOMINAL, ORDINAL, NUMERIC, 'regression')
clf_cv = cv_compare(train_df, NOMINAL, ORDINAL, NUMERIC, 'classification')
display(reg_cv)
display(clf_cv)

c:\Users\laura\OneDrive\Laura\Uni\3년\2학기\Machine Learning\Final Project\seoul_foreigners_public_release\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\laura\OneDrive\Laura\Uni\3년\2학기\Machine Learning\Final Project\seoul_foreigners_public_release\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\laura\OneDrive\Laura\Uni\3년\2학기\Machine Learning\Final Project\seoul_foreigners_public_release\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifie

,model,task,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std
0,SVR_RBF,regression,0.949864,0.035931,0.708531,0.031329,0.289980,0.030325
1,RandomForest,regression,0.954522,0.030320,0.711151,0.031176,0.282939,0.024744
2,LinearRegression,regression,1.018073,0.036058,0.780721,0.018102,0.182608,0.061647
3,DummyMean,regression,1.128106,0.034015,0.865443,0.028179,-0.001053,0.000659


,model,task,cv_f1_mean,cv_f1_std,cv_f1_macro_mean,cv_f1_macro_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_accuracy_mean,cv_accuracy_std,cv_roc_auc_mean,cv_roc_auc_std
0,SVC_RBF,classification,0.800832,0.007397,0.658865,0.013239,0.651665,0.012036,0.7180,0.010173,0.760267,0.011651
1,LogisticRegression,classification,0.794799,0.011501,0.657187,0.016422,0.650553,0.015116,0.7125,0.014748,0.732039,0.015196
2,RandomForest,classification,0.808275,0.007351,0.616518,0.020007,0.616943,0.015590,0.7125,0.011726,0.768747,0.014159
3,DummyMostFrequent,classification,0.791541,0.000000,0.395770,0.000000,0.500000,0.000000,0.6550,0.000000,0.500000,0.000000


## 4. Winner selection without consulting test

In [5]:
reg_winner, clf_winner = reg_cv.loc[0, 'model'], clf_cv.loc[0, 'model']
print('Selected regression:', reg_winner)
print('Selected classification:', clf_winner)

Selected regression: SVR_RBF
Selected classification: SVC_RBF


## 5. Shared test-set evaluation (descriptive; prior test reuse disclosed)

In [6]:
reg_model, reg_test, _ = evaluate_winner(train_df, test_df, NOMINAL, ORDINAL, NUMERIC, 'regression', reg_winner)
clf_model, clf_test, confusion = evaluate_winner(train_df, test_df, NOMINAL, ORDINAL, NUMERIC, 'classification', clf_winner)
display(reg_test)
display(clf_test)
display(confusion)

c:\Users\laura\OneDrive\Laura\Uni\3년\2학기\Machine Learning\Final Project\seoul_foreigners_public_release\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,model,task,mae,rmse,r2
0,SVR_RBF,regression,0.705652,0.995645,0.235375


,model,task,accuracy,precision,recall,f1,f1_macro,balanced_accuracy,roc_auc
0,SVC_RBF,classification,0.736,0.755208,0.884146,0.814607,0.678137,0.668817,0.750833


,pred_0,pred_1
actual_0,78,94
actual_1,38,290


## 6. Interpretation using a training-only validation partition

Permutation importance refits the selected estimator on an 80% subset of training and evaluates importance on the remaining training subset. This is not causal importance and does not use the common test sample to select features.

In [7]:
reg_imp = feature_permutation(reg_model, train_df, FEATURES, 'regression')
clf_imp = feature_permutation(clf_model, train_df, FEATURES, 'classification')
display(reg_imp.head(15))
display(clf_imp.head(15))

c:\Users\laura\OneDrive\Laura\Uni\3년\2학기\Machine Learning\Final Project\seoul_foreigners_public_release\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,feature,importance_mean,importance_std
18,Q15,0.050499,0.006746
20,Q6_2,0.015666,0.004014
40,SQ6_2_log1p,0.012044,0.001739
32,Q24_2,0.011078,0.000801
15,Q20_3,0.010514,0.008840
17,DQ4_1,0.010249,0.002859
33,Q24_3,0.009611,0.001499
39,SQ6_1_log1p,0.009352,0.005504
26,Q6_8,0.008053,0.003024
6,DQ2,0.007213,0.003072


,feature,importance_mean,importance_std
14,Q20_2,0.047433,0.013942
39,SQ6_1_log1p,0.031150,0.005184
38,korean_proficiency_10,0.026668,0.008540
18,Q15,0.023913,0.011961
35,Q24_5,0.020840,0.007205
23,Q6_5,0.019561,0.005269
15,Q20_3,0.019522,0.007216
10,Q19_2,0.019338,0.009703
17,DQ4_1,0.018439,0.005808
20,Q6_2,0.017708,0.013265


## 7. Export reproducible metrics

Review all results in `findings/05_global_ml_model.md`. Do not replace the results with pre-correction numbers. In the report, compare the integrated model with thematic models as *descriptive exploratory comparisons* because the same test subjects were already inspected.

In [8]:
for stem,table in {
    '05_global_regression_cv': reg_cv,
    '05_global_classification_cv': clf_cv,
    '05_global_regression_test': reg_test,
    '05_global_classification_test': clf_test,
    '05_global_confusion_matrix': confusion,
    '05_global_regression_importance': reg_imp,
    '05_global_classification_importance': clf_imp,
    '05_global_feature_audit': feature_quality,
}.items():
    table.to_csv(RESULTS_DIR / f'{stem}.csv', index=stem == '05_global_confusion_matrix')
print('Integrated results saved to:', RESULTS_DIR)

Integrated results saved to: C:\Users\laura\OneDrive\Laura\Uni\3년\2학기\Machine Learning\Final Project\seoul_foreigners_public_release\results


## 8. Limitations and next steps

1. The survey is cross-sectional and self-reported; associations do not imply cause and effect.
2. Survey skip patterns can be predictive in their own right; missing indicators should not be interpreted as specific lived experiences.
3. The 500-person test set was examined during earlier thematic analyses, so an independent replication or an external survey year would be required for an unbiased final confirmation.
4. The 8-point classification cut-off is a declared operational definition, not a clinical or validated diagnostic threshold.
5. Survey sampling weights have not been incorporated; models target sample-level prediction and cannot automatically be interpreted as population-representative estimates.
6. The same model hyperparameters are used across thematic and integrated comparisons. Hyperparameter optimization is a future training-CV-only direction, not part of this benchmark.